# CVE2Patch — Raccolta e ranking preliminare

**Fase di discovery — versione 4 (codice 4.1, refactoring).** Obiettivo: un primo pool di CVE Linux con contesto pubblico utile alla futura valutazione lato host. Le evidenze raccolte non sono test di riproducibilità o di detection.

### Fondamento metodologico
| Riferimento | Scelta adottata nel notebook |
|---|---|
| [CVEfixes — Bhandari et al., 2021](https://doi.org/10.1145/3475960.3475985) | Acquisizione automatizzata NVD e associazione di dati esterni ai record CVE. |
| [The Anatomy of a Vulnerability Database, 2023](https://doi.org/10.1016/j.jss.2023.111679) | Integrazione di informazioni da più database, qui mediante CVE ID. |
| [Jacobs et al., 2020](https://doi.org/10.1093/cybsec/tyaa015) | Distinzione tra fonti di exploit con copertura diversa. Non adottiamo il modello di previsione del rischio. |
| [Mu et al., 2018](https://www.usenix.org/conference/usenixsecurity18/presentation/mu) | Dare rilievo alla disponibilità delle informazioni utili alla riproduzione; la presenza di documentazione non ne prova il successo. |
| [CVE-GENIE — Ullah et al., 2025; v2 2026](https://arxiv.org/abs/2509.01835v2) | Raccogliere risorse ed evidenze per preparare il contesto prima della futura fase agentica; nessuna ricostruzione o generazione in questa fase. |

**Scelte specifiche CVE2Patch (riunione, tesi5.txt):** periodo dal 2019, Linux, AV Network/Local, kernel separato, match deterministici su fonti prestabilite, ranking a contributi decrescenti, selezione per soglia e revisione manuale. I paper motivano il metodo, non validano i pesi. Il punteggio è un indice documentale, non CVSS, EPSS, probabilità o efficacia di Falco.

Eseguire dalla cartella originale del notebook. I file in `CVE2Patch/data/raw/` vengono riutilizzati; gli output derivati vengono rigenerati; i due nuovi indici PoC sono scaricati solo se mancanti. Nessun exploit viene scaricato o eseguito. I feed annuali NVD sono per anno dell'identificativo: il filtro sulla pubblicazione vale per i record acquisiti, ma non recupera eventuali ID anteriori al 2019 pubblicati dal 2019 in poi.


## 1. Librerie

Dipendenze per Python 3.11+ (requests, pandas, PyYAML). Funzioni comuni per il download con cache, gli identificativi CVE, il salvataggio delle tabelle e i riepiloghi. Le celle delle fonti conservano i rispettivi parser.


In [ ]:
import requests
import pandas as pd
import json
import re
import tomllib
import yaml
import hashlib
import gzip
import zipfile
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from urllib.parse import urlparse, unquote
from IPython.display import display


# Funzioni comuni: download, CVE ID, tabelle e riepiloghi.
CVE_PATTERN = re.compile(r"CVE-\d{4}-\d{4,}", re.I)


def cached_download(url, path, timeout=120, validate=None):
    """Riutilizza il file locale; un download incompleto non diventa cache."""
    if path.exists():
        if validate is not None and not validate(path):
            raise ValueError(f"File non valido: {path}. Rimuoverlo e rieseguire.")
        return path
    tmp = path.with_name(path.name + ".part")
    try:
        print(f"Download {path.name}...")
        with requests.get(url, stream=True, timeout=timeout) as response:
            response.raise_for_status()
            with tmp.open("wb") as f:
                for chunk in response.iter_content(1024 * 1024):
                    f.write(chunk)
        if validate is not None and not validate(tmp):
            raise ValueError(f"Formato non valido: {url}")
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)
    return path


def cve_ids(text):
    return sorted({m.upper() for m in CVE_PATTERN.findall(str(text))})


def records_for_ids(ids, fields):
    return ({"cve_id": cve_id, **fields} for cve_id in ids)


def save_table(records, filename, columns=None, deduplicate=True):
    frame = pd.DataFrame(records, columns=columns)
    if deduplicate:
        frame = frame.drop_duplicates()
    frame.to_csv(PROCESSED_DIR / filename, index=False)
    return frame


def report_matches(frame):
    ids = set(frame["cve_id"])
    print(f"Corrispondenze con candidate: {df_candidates['cve_id'].isin(ids).sum()}")
    print(f"Possibili CVE da recuperare: {df_unknown['cve_id'].isin(ids).sum()}")

print("Librerie e funzioni comuni disponibili.")


## 2. Configurazione
Un'unica cella per le date, gli attack vector, i pesi e la soglia minima inclusiva. Il perimetro rimane Linux; il kernel è sempre separato. I pesi sono empirici, non probabilità. La configurazione deve mantenere un massimo teorico pari a 100 e rispettare la gerarchia delle fonti. Non esiste un limite numerico al pool.


In [ ]:
BASE_DIR = Path.cwd() / "CVE2Patch"
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "output"
for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START_DATE = "2019-01-01"
END_DATE = "2026-11-17"  # Cut-off esplicito; modificabile prima dell'esecuzione
ATTACK_VECTORS = ["NETWORK", "LOCAL"]
MIN_SCORE = 60

WEIGHTS = {
    "vulhub": 30,
    "msf_min": 30,
    "msf_max": 40,
    "edb_verified": 30,
    "edb_unverified": 25,
    "poc_primary": 20,
    "edb_bonus": 5,
    "poc_bonus": 5,
    "falco": 20,
}
assert datetime.fromisoformat(START_DATE) <= datetime.fromisoformat(END_DATE)
assert ATTACK_VECTORS and set(ATTACK_VECTORS) <= {"NETWORK", "LOCAL"}
assert isinstance(MIN_SCORE, (int, float)) and 0 <= MIN_SCORE <= 100
assert all(isinstance(w, (int, float)) and 0 <= w <= 100 for w in WEIGHTS.values())
assert WEIGHTS["msf_max"] >= WEIGHTS["msf_min"] >= WEIGHTS["edb_verified"] >= WEIGHTS["edb_unverified"] >= WEIGHTS["poc_primary"]
assert WEIGHTS["edb_bonus"] <= WEIGHTS["edb_unverified"] and WEIGHTS["poc_bonus"] <= WEIGHTS["poc_primary"]
MAX_EXPLOIT = WEIGHTS["msf_max"] + WEIGHTS["edb_bonus"] + WEIGHTS["poc_bonus"]
MAX_SCORE = MAX_EXPLOIT + WEIGHTS["vulhub"] + WEIGHTS["falco"]
assert abs(MAX_SCORE - 100) < 1e-8, "Ribilanciare i pesi: il massimo deve essere 100."
print(f"Periodo: {START_DATE} - {END_DATE} | Soglia: {MIN_SCORE}/100")
print(f"Directory: {BASE_DIR}")


## 3. NVD — Acquisizione dei feed

Download dei soli file annuali mancanti; i file compressi preesistenti vengono riutilizzati.

In [ ]:
NVD_DIR = RAW_DIR / "nvd_feeds"
NVD_DIR.mkdir(parents=True, exist_ok=True)
NVD_BASE_URL = "https://nvd.nist.gov/feeds/json/cve/2.0"
NVD_FILES = []
for year in range(int(START_DATE[:4]), int(END_DATE[:4]) + 1):
    filename = f"nvdcve-2.0-{year}.json.gz"
    path = NVD_DIR / filename
    if path.exists():
        print(f"Dataset {year} già disponibile.")
    NVD_FILES.append(cached_download(f"{NVD_BASE_URL}/{filename}", path))
print(f"\nDataset annuali disponibili: {len(NVD_FILES)}")


## 4. NVD — Estrazione e filtri fondamentali
Periodo e stato, poi classificazione Linux via CPE (prodotto/piattaforma vulnerabile o requisito ambientale), separazione kernel e filtro AV. Le semplici menzioni Linux non ammettono automaticamente una CVE. Si conservano i casi incerti e quelli non Linux per i controlli; il kernel è conservato anche con AV fuori perimetro. Le configurazioni CPE complesse e l'identificazione kernel restano euristiche. I tag dei riferimenti vengono conservati, senza punti.


In [ ]:
def extract_cvss(cve):
    for version in ["cvssMetricV40", "cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
        entries = cve.get("metrics", {}).get(version, [])
        if entries:
            metric = next((m for m in entries if m.get("type") == "Primary"), entries[0])
            data = metric.get("cvssData", {})
            return data.get("baseScore"), data.get("attackVector", data.get("accessVector")), data.get("version")
    return None, None, None

LINUX_PRODUCTS = {"linux_kernel", "linux", "ubuntu_linux", "ubuntu", "debian_linux", "debian",
                  "fedora", "centos", "rocky_linux", "almalinux", "amazon_linux", "amazon_linux_2",
                  "amazon_linux_2023", "arch_linux", "opensuse", "leap", "tumbleweed", "oracle_linux",
                  "raspberry_pi_os", "kali_linux", "red_hat_enterprise_linux"}
NONLINUX = re.compile(r"^(windows(?:_|$)|macos(?:_|$)|mac_os(?:_|$)|os_x(?:_|$)|ios(?:_|$)|ipados(?:_|$)|android(?:_|$)|freebsd(?:_|$)|openbsd(?:_|$)|netbsd(?:_|$)|solaris(?:_|$)|aix(?:_|$))")
LINUX_TEXT = re.compile(r"\b(?:linux|ubuntu|debian|fedora|centos|red hat|suse|almalinux|rocky linux)\b", re.I)
KERNEL_TEXT = re.compile(r"(?:^\s*(?:in the |the )?linux kernel\b|\b(?:flaw|vulnerability|bug|issue|race condition|use-after-free)\b[^.]{0,65}\b(?:in|within) (?:the )?linux kernel\b)", re.I)


def parse_cpe(match):
    criteria = match.get("criteria", "")
    parts = re.split(r"(?<!\\):", criteria.lower())
    if len(parts) != 13 or parts[:2] != ["cpe", "2.3"]:
        return None
    part, vendor, product, target_sw = parts[2], parts[3], parts[4], parts[10]
    linux = (part == "o" and (product in LINUX_PRODUCTS or product.startswith(("enterprise_linux", "linux_enterprise", "ubuntu_linux", "debian_linux")))) or target_sw == "linux"
    other = (part == "o" and bool(NONLINUX.match(product))) or bool(NONLINUX.match(target_sw))
    return {"criteria": criteria, "vulnerable": match.get("vulnerable") is True,
            "linux": linux, "other": other and not linux, "part": part,
            "kernel": part == "o" and vendor == "linux" and product == "linux_kernel"}


def walk_cpe(obj):
    # Trascuriamo rami negati. Le configurazioni complesse rimangono euristiche.
    if not isinstance(obj, dict) or obj.get("negate", False):
        return [], False
    terms = [p for m in obj.get("cpeMatch", []) if (p := parse_cpe(m)) is not None]
    context = False
    for child in obj.get("nodes", []) + obj.get("children", []):
        child_terms, child_context = walk_cpe(child)
        terms.extend(child_terms)
        context |= child_context
    # Un requisito ambientale Linux conta solo in una congiunzione con software vulnerabile.
    if str(obj.get("operator", "OR")).upper() == "AND":
        context |= (any(t["linux"] and not t["vulnerable"] for t in terms)
                    and any(t["vulnerable"] and t["part"] == "a" and not t["other"] for t in terms))
    return terms, context


def platform_evidence(cve, description):
    terms, linux_context = [], False
    for configuration in cve.get("configurations", []):
        group, context = walk_cpe(configuration)
        terms.extend(group)
        linux_context |= context
    affected = [t for t in terms if t["vulnerable"]]
    linux_direct = any(t["linux"] for t in affected)
    other_only = bool(affected) and all(t["other"] for t in affected) and not linux_context
    kernel = (any(t["kernel"] for t in affected)
              or cve.get("sourceIdentifier", "").lower() == "cve@kernel.org"
              or bool(KERNEL_TEXT.search(description)))
    linux_text = bool(LINUX_TEXT.search(description))
    if kernel:
        status = "kernel"
    elif linux_direct:
        status = "linux_direct"
    elif linux_context:
        status = "linux_context"
    elif other_only:
        status = "nonlinux_cpe"
    else:
        status = "unknown"
    return {"platform_nvd": status, "linux_direct": linux_direct, "linux_context": linux_context,
            "linux_text": linux_text, "kernel": kernel,
            "cpe": json.dumps(sorted({t["criteria"] for t in terms})),
            "vulnerable_cpe": json.dumps(sorted({t["criteria"] for t in affected}))}


records = []
for file_path in NVD_FILES:
    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    for item in data["vulnerabilities"]:
        cve = item["cve"]
        description = next((d["value"] for d in cve.get("descriptions", []) if d.get("lang") == "en"), "")
        cvss, attack_vector, cvss_version = extract_cvss(cve)
        cwes = sorted({d["value"] for w in cve.get("weaknesses", []) for d in w.get("description", [])})
        references = sorted({r["url"] for r in cve.get("references", []) if r.get("url")})
        reference_details = [{"url": r["url"], "source": r.get("source"), "tags": r.get("tags", [])}
                             for r in cve.get("references", []) if r.get("url")]
        records.append({"cve_id": cve["id"], "published": cve.get("published"),
                        "description": description, "status": cve.get("vulnStatus"),
                        "cvss": cvss, "cvss_version": cvss_version, "attack_vector": attack_vector,
                        "cwe": json.dumps(cwes), "references": json.dumps(references),
                        "reference_details": json.dumps(reference_details, ensure_ascii=False),
                        **platform_evidence(cve, description)})
    print(f"Letto {file_path.name}")
    del data

df_all = pd.DataFrame(records).drop_duplicates("cve_id").reset_index(drop=True)
del records
in_period = df_all["published"].fillna("").str[:10].between(START_DATE, END_DATE)
df_valid = df_all[in_period & df_all["status"].ne("Rejected")].copy()
# Dopo periodo e stato, classificazione Linux. Il kernel è un ramo separato.
df_linux_pre_av = df_valid[df_valid["platform_nvd"].isin(["linux_direct", "linux_context"])].copy()
df_unknown_pre_av = df_valid[df_valid["platform_nvd"].eq("unknown")].copy()
df_nonlinux_pre_av = df_valid[df_valid["platform_nvd"].eq("nonlinux_cpe")].copy()
df_kernel = df_valid[df_valid["kernel"]].copy()
assert len(df_valid) == sum(map(len, [df_linux_pre_av, df_unknown_pre_av, df_nonlinux_pre_av, df_kernel]))

def allowed_av(frame):
    return frame[frame["attack_vector"].isin(ATTACK_VECTORS)].copy()

df_candidates = allowed_av(df_linux_pre_av)
df_unknown = allowed_av(df_unknown_pre_av)
df_nonlinux = allowed_av(df_nonlinux_pre_av)
df_non_kernel = df_valid[~df_valid["kernel"]].copy()
df_av = pd.concat([df_candidates, df_unknown, df_nonlinux], ignore_index=True)
df_av_excluded = df_non_kernel[~df_non_kernel["attack_vector"].isin(ATTACK_VECTORS)].copy()
assert len(df_av) + len(df_av_excluded) + len(df_kernel) == len(df_valid)

for name, frame in {"all_cves": df_all, "kernel_cves": df_kernel,
                    "candidates": df_candidates, "platform_unknown": df_unknown,
                    "platform_nonlinux": df_nonlinux, "attack_vector_excluded": df_av_excluded}.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)

print(f"Record NVD unici: {len(df_all)}")
print(f"Non rigettate nel periodo: {len(df_valid)}")
print("Classificazione piattaforma prima dell'AV:")
print(df_valid["platform_nvd"].value_counts())
print(f"Kernel separato (tutti gli AV): {len(df_kernel)}")
print(f"Linux NVD con AV ammesso: {len(df_candidates)}")
print(f"Piattaforma incerta con AV ammesso: {len(df_unknown)}")
print(f"Non Linux con AV ammesso: {len(df_nonlinux)}")
print(f"Non kernel con AV escluso: {len(df_av_excluded)}")


## 5. Vulhub — Ambienti vulnerabili

Indice degli ambienti disponibili; una corrispondenza non equivale a un laboratorio testato.

In [ ]:
VULHUB_URL = "https://raw.githubusercontent.com/vulhub/vulhub/master/environments.toml"
vulhub_file = cached_download(VULHUB_URL, RAW_DIR / "vulhub_environments.toml", timeout=60)
with vulhub_file.open("rb") as f:
    vulhub_data = tomllib.load(f)

vulhub_records = []
for env in vulhub_data["environment"]:
    fields = {
        "vulhub_name": env.get("name"), "vulhub_app": env.get("app"),
        "vulhub_path": env.get("path"), "vulhub_tags": json.dumps(env.get("tags", [])),
        "vulhub_url": "https://github.com/vulhub/vulhub/tree/master/" + env.get("path", ""),
    }
    vulhub_records.extend(records_for_ids((c.upper() for c in env.get("cve", [])), fields))

df_vulhub = save_table(vulhub_records, "vulhub.csv", columns=[
    "cve_id", "vulhub_name", "vulhub_app", "vulhub_path", "vulhub_tags", "vulhub_url"
], deduplicate=False)  # Preserva le associazioni originali dell'indice.
print(f"Ambienti Vulhub: {len(vulhub_data['environment'])}")
print(f"CVE uniche Vulhub: {df_vulhub['cve_id'].nunique()}")
report_matches(df_vulhub)
display(df_vulhub.head())


## 6. Metasploit — Metadati dei moduli
Solo i moduli di tipo exploit sono utilizzabili nel ranking; si conservano piattaforma e rank ufficiale. La compatibilità è calcolata nella sezione 10. Se più moduli sono compatibili, si sceglie il rank valido maggiore.


In [ ]:
MSF_URL = "https://raw.githubusercontent.com/rapid7/metasploit-framework/master/db/modules_metadata_base.json"
msf_file = cached_download(MSF_URL, RAW_DIR / "metasploit_metadata.json")
msf_data = json.loads(msf_file.read_text(encoding="utf-8"))

msf_records = []
for module in msf_data.values():
    references = module.get("references", [])
    fields = {
        "msf_name": module.get("name"), "msf_module": module.get("fullname"),
        "msf_type": module.get("type"), "msf_platform": module.get("platform"),
        "msf_rank": module.get("rank"), "msf_description": module.get("description"),
        "msf_references": json.dumps(references, ensure_ascii=False),
    }
    msf_records.extend(records_for_ids(cve_ids(json.dumps(references)), fields))

df_msf = save_table(msf_records, "metasploit.csv", columns=[
    "cve_id", "msf_name", "msf_module", "msf_type", "msf_platform", "msf_rank", "msf_description", "msf_references"
])
df_msf_exploits = df_msf[df_msf["msf_type"].eq("exploit")].copy()
print(f"Moduli Metasploit: {len(msf_data)}")
print(f"CVE uniche: {df_msf['cve_id'].nunique()}")
print(f"CVE associate a exploit: {df_msf_exploits['cve_id'].nunique()}")
report_matches(df_msf_exploits)
display(df_msf.head())


## 7. Exploit-DB — Metadati
Associazione via CVE ID, conservando la piattaforma e lo stato `verified`. La verifica dichiarata dalla fonte non dimostra la riproducibilità sul nostro laboratorio. I record non verificati non sono scartati.


In [ ]:
EDB_URL = "https://gitlab.com/exploit-database/exploitdb/-/raw/main/files_exploits.csv"
edb_file = cached_download(EDB_URL, RAW_DIR / "exploitdb.csv")
edb_data = pd.read_csv(edb_file).fillna("")

# Mappa esplicita dei campi CSV da conservare.
EDB_FIELDS = {"edb_id": "id", "edb_description": "description", "edb_type": "type",
              "edb_platform": "platform", "edb_verified": "verified", "edb_date": "date_published",
              "edb_file": "file"}
edb_records = []
for row in edb_data.to_dict("records"):
    fields = {field: row.get(original) for field, original in EDB_FIELDS.items()}
    edb_records.extend(records_for_ids(cve_ids(row.get("codes", "")), fields))

df_edb = save_table(edb_records, "exploitdb.csv", columns=["cve_id", *EDB_FIELDS])
print(f"Exploit indicizzati: {len(edb_data)}")
print(f"CVE uniche: {df_edb['cve_id'].nunique()}")
report_matches(df_edb)
display(df_edb.head())


## 8. GitHub e riferimenti PoC
I riferimenti GitHub di NVD restano informativi. Per i potenziali PoC si consultano due indici pubblici già esistenti: nomi-sec/PoC-in-GitHub (JSON) e trickest/cve (Markdown). Si leggono soltanto gli indici, non si seguono i link verso il codice. I due archivi sono scaricati una volta e riutilizzati. I riferimenti indicizzati possono essere falsi positivi: si richiede un'indicazione esplicita PoC/exploit e l'associazione alla CVE specifica; i riferimenti generici rimangono documentali. Duplicati URL e duplicati evidenti delle altre fonti non danno punti aggiuntivi. Se un indice non è disponibile o non è riconoscibile, la cella si interrompe esplicitamente, evitando punteggi incompleti presentati come definitivi.


In [ ]:
GITHUB_TYPES = {"advisories": "advisory", "commit": "commit", "issues": "issue", "pull": "pull_request"}
github_records = []
df_search = pd.concat([df_av, df_kernel], ignore_index=True)[["cve_id", "references"]].drop_duplicates("cve_id")
for row in df_search.itertuples(index=False):
    for url in json.loads(row.references):
        try:
            parsed = urlparse(url)
            hostname = parsed.hostname
        except ValueError:
            continue
        parts = parsed.path.strip("/").split("/")
        if hostname not in ["github.com", "www.github.com"] or len(parts) < 2:
            continue
        repository = None if parts[0] == "advisories" else "/".join(parts[:2])
        fallback = "repository" if len(parts) == 2 else "file" if "blob" in parts else "other"
        reference_type = next((kind for segment, kind in GITHUB_TYPES.items() if segment in parts), fallback)
        github_records.append({"cve_id": row.cve_id, "github_repository": repository,
                               "github_url": url, "github_type": reference_type})

df_github = save_table(github_records, "github_references.csv", columns=[
    "cve_id", "github_repository", "github_url", "github_type"
])
print(f"Riferimenti GitHub di NVD: {len(df_github)} (informativi)")

# Archivi degli indici: nessuna richiesta verso i repository dei PoC.
POC_INDEXES = {
    "nomi-sec": ("https://codeload.github.com/nomi-sec/PoC-in-GitHub/zip/refs/heads/master", "poc_nomi_sec.zip", ".json"),
    "trickest": ("https://codeload.github.com/trickest/cve/zip/refs/heads/main", "poc_trickest.zip", ".md"),
}

def cached_zip(url, path):
    try:
        return cached_download(url, path, timeout=(20, 180), validate=zipfile.is_zipfile)
    except Exception as exc:
        raise RuntimeError(f"Indice PoC non disponibile: {url}. Rieseguire la cella 8; nessun ranking V4 completo è prodotto.") from exc

POC_CUE = re.compile(r"(?<![a-z0-9])(?:poc|proof[-_ ]?of[-_ ]?concept|exploits?)(?![a-z0-9])", re.I)
URL_PATTERN = re.compile(r"https?://[^\s<>\]\)\"']+", re.I)

def normalize_url(url):
    try:
        p = urlparse(str(url).strip().rstrip(".,;"))
        if p.scheme not in {"http", "https"} or not p.hostname:
            return None
        host = p.hostname.lower().removeprefix("www.")
        path = unquote(p.path).rstrip("/")
        if host == "github.com":
            path = path.lower().removesuffix(".git")
        return host + path + ("?" + p.query if p.query else "")
    except ValueError:
        return None

# Riconoscimento conservativo: non basta che il repository nomini la CVE.
# Per nomi-sec usiamo nome/descrizione/topics; per trickest il riferimento stesso.
def explicit_poc(cve_id, text):
    text = unquote(str(text))
    return bool(POC_CUE.search(text) and cve_id in cve_ids(text))

def trickest_links(text):
    # Lo schema pubblico contiene ### POC, #### Reference, #### Github.
    # Non si leggono link nelle descrizioni o in altre sezioni.
    in_poc = False
    for line in text.splitlines():
        head = re.match(r"^\s*(#{1,6})\s+(.+?)\s*$", line)
        if head:
            level, label = len(head[1]), head[2].strip().casefold()
            if level <= 3:
                in_poc = label in {"poc", "proof of concept", "proof-of-concept"}
            continue
        if in_poc and not line.lstrip().startswith("!["):
            for url in URL_PATTERN.findall(line):
                yield url.rstrip(".,;")

wanted_ids = set(df_candidates["cve_id"]) | set(df_unknown["cve_id"]) | set(df_kernel["cve_id"])
poc_records = []
poc_input_files = []
poc_index_status = {}
for source, (url, filename, extension) in POC_INDEXES.items():
    path = cached_zip(url, RAW_DIR / filename)
    poc_input_files.append(path)
    matched_files = parsed_files = recognized_sections = 0
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            entry = PurePosixPath(member.filename)
            if entry.suffix != extension or not CVE_PATTERN.fullmatch(entry.stem):
                continue
            cve_id = entry.stem.upper()
            if cve_id not in wanted_ids:
                continue
            matched_files += 1
            raw = archive.read(member).decode("utf-8")
            if source == "nomi-sec":
                items = json.loads(raw)
                if not isinstance(items, list):
                    raise ValueError(f"Formato nomi-sec inatteso in {entry}")
                for item in items:
                    if not isinstance(item, dict):
                        raise ValueError(f"Record nomi-sec inatteso in {entry}")
                    link = item.get("html_url")
                    if not isinstance(link, str):
                        continue
                    text = " ".join(str(item.get(k) or "") for k in ("full_name", "name", "description", "topics"))
                    poc_records.append((cve_id, source, link, explicit_poc(cve_id, text)))
            else:
                if re.search(r"(?im)^#{1,3}\s+POC\s*$", raw):
                    recognized_sections += 1
                for link in trickest_links(raw):
                    poc_records.append((cve_id, source, link, explicit_poc(cve_id, link)))
            parsed_files += 1
    if wanted_ids and not matched_files:
        print(f"ATTENZIONE: {source}, nessun file per gli ID richiesti; verificare la copertura.")
    if source == "trickest" and matched_files and not recognized_sections:
        raise ValueError("Formato Trickest non riconosciuto: controllare le intestazioni POC.")
    poc_index_status[source] = {"url": url, "status": "consulted", "matched_files": matched_files,
                                "parsed_files": parsed_files, "poc_sections": recognized_sections if source == "trickest" else None}
    print(f"{source}: {parsed_files} file di indice letti")

df_poc = pd.DataFrame(poc_records, columns=["cve_id", "poc_index", "poc_url", "explicit_poc_hint"])
df_poc["normalized_url"] = df_poc["poc_url"].map(normalize_url)
df_poc = df_poc.dropna(subset=["normalized_url"]).drop_duplicates(["cve_id", "poc_index", "normalized_url"])
# Conserviamo tutti i riferimenti degli indici; la decisione sui bonus è centralizzata nella cella 10.
df_poc.to_csv(PROCESSED_DIR / "poc_index_references.csv", index=False)
print(f"Riferimenti dagli indici: {len(df_poc)}; con indizio specifico PoC: {df_poc['explicit_poc_hint'].sum()}")


## 9. Falco — Riferimenti pubblici
Regole Stable, Incubating e Sandbox: match esplicito del CVE ID in una regola. Al punteggio contribuisce al massimo una regola con sorgente syscall, a prescindere dalla maturità. `enabled`, priorità e maturità restano informativi. Non si raccolgono ancora log o file scap; nessuna rilevazione sperimentale.


In [ ]:
FALCO_BASE_URL = "https://raw.githubusercontent.com/falcosecurity/rules/main/rules/"
FALCO_FILES = {"stable": "falco_rules.yaml", "incubating": "falco-incubating_rules.yaml",
               "sandbox": "falco-sandbox_rules.yaml"}
falco_records, total_rules = [], 0
for maturity, filename in FALCO_FILES.items():
    path = cached_download(FALCO_BASE_URL + filename, RAW_DIR / filename, timeout=60)
    rules = yaml.safe_load(path.read_text(encoding="utf-8"))
    for rule in rules:
        if not isinstance(rule, dict) or "rule" not in rule:
            continue
        total_rules += 1
        fields = {
            "falco_rule": rule.get("rule"), "falco_description": rule.get("desc"),
            "falco_priority": rule.get("priority"), "falco_maturity": maturity,
            "falco_enabled": rule.get("enabled"), "falco_source": rule.get("source", "syscall"),
            "falco_url": "https://github.com/falcosecurity/rules/blob/main/rules/" + filename,
        }
        falco_records.extend(records_for_ids(cve_ids(json.dumps(rule)), fields))

df_falco = save_table(falco_records, "falco_rules.csv", columns=[
    "cve_id", "falco_rule", "falco_description", "falco_priority", "falco_maturity",
    "falco_enabled", "falco_source", "falco_url"
])
print(f"Regole Falco analizzate: {total_rules}")
print(f"Associazioni CVE-regola: {len(df_falco)}")
print(f"CVE uniche: {df_falco['cve_id'].nunique()}")
report_matches(df_falco)
display(df_falco.head())


## 10. Integrazione e recupero
Incrocio di tutti gli indici per CVE ID. Si recuperano provvisoriamente le piattaforme incerte con Vulhub o artefatto esplicitamente Linux; i conflitti non sono recuperati automaticamente. Si valutano centralmente i singoli moduli e gli exploit. Il kernel viene arricchito a parte. Il PoC è un riferimento candidato, non codice verificato; URL uguali, repository delle fonti già indicizzate e riferimenti identici nei moduli sono esclusi dal bonus per prudenza. Statistiche e motivi di esclusione sono ricostruibili.


In [ ]:
# Classificazione conservativa della piattaforma dei metadati.
PORTABLE_PLATFORMS = {"multi", "multiple", "unix", "posix", "webapps", "webapp", "php", "java", "jsp", "python", "ruby", "perl", "cgi", "nodejs", "node", "javascript"}
OTHER_PLATFORMS = {"windows", "win", "win32", "win64", "osx", "macos", "android", "ios", "freebsd", "openbsd", "netbsd", "solaris", "aix", "netware"}
def platform_fit(value):
    tokens = set(re.findall(r"[a-z0-9]+", str(value).lower()))
    if "linux" in tokens:
        return "linux"
    if tokens & OTHER_PLATFORMS:
        return "other_or_mixed"
    if tokens & PORTABLE_PLATFORMS:
        return "portable"
    return "unknown"

msf_platform = df_msf_exploits["msf_platform"].map(platform_fit)
edb_platform = df_edb["edb_platform"].map(platform_fit)
msf_linux_ids = set(df_msf_exploits.loc[msf_platform.eq("linux"), "cve_id"])
edb_linux_ids = set(df_edb.loc[edb_platform.eq("linux"), "cve_id"])
msf_compatible_rows = df_msf_exploits.loc[msf_platform.isin(["linux", "portable"])].copy()
edb_compatible_rows = df_edb.loc[edb_platform.isin(["linux", "portable"])].copy()
msf_compatible_ids = set(msf_compatible_rows["cve_id"])
edb_compatible_ids = set(edb_compatible_rows["cve_id"])
vulhub_ids = set(df_vulhub["cve_id"])
falco_syscall_ids = set(df_falco.loc[df_falco["falco_source"].eq("syscall"), "cve_id"])

# Falco e GitHub da soli non attestano la piattaforma della CVE.
recovery_ids = vulhub_ids | msf_linux_ids | edb_linux_ids
# Non recuperiamo automaticamente i casi con CPE esclusivamente non Linux.
df_recovered = df_unknown[df_unknown["cve_id"].isin(recovery_ids)].copy()
df_conflicts = df_nonlinux[df_nonlinux["cve_id"].isin(recovery_ids | falco_syscall_ids)].copy()
df_pending = df_unknown[~df_unknown["cve_id"].isin(recovery_ids)].copy()
df_initial = df_candidates.copy()
df_initial["platform_basis"] = df_initial["platform_nvd"]
df_recovered["platform_basis"] = "external_provisional"

df_enriched = pd.concat([df_initial, df_recovered], ignore_index=True)
assert df_enriched["cve_id"].is_unique and not df_enriched["kernel"].any()

rank = pd.to_numeric(msf_compatible_rows["msf_rank"], errors="coerce")
msf_compatible_rows["rank_valid"] = rank.where(rank.isin(range(0, 601, 100)))
msf_ranks = msf_compatible_rows.groupby("cve_id")["rank_valid"].max()
edb_flag = edb_compatible_rows["edb_verified"].fillna("").astype(str).str.strip().str.lower()
edb_compatible_rows["verified_flag"] = edb_flag.isin(["1", "1.0", "true", "yes"])
edb_compatible_rows["verified_known"] = edb_flag.isin(["1", "1.0", "true", "yes", "0", "0.0", "false", "no"])
edb_verified_ids = set(edb_compatible_rows.loc[edb_compatible_rows["verified_flag"], "cve_id"])
edb_known_ids = set(edb_compatible_rows.loc[edb_compatible_rows["verified_known"], "cve_id"])

# Duplicati evidenti: sorgenti già indicizzate, URL uguali a riferimenti MSF compatibili.
# Non si tenta di stabilire l'identità del codice o l'indipendenza effettiva delle evidenze.
DUP_REPOS = ("github.com/rapid7/metasploit-framework", "github.com/offensive-security/exploitdb",
             "gitlab.com/exploit-database/exploitdb", "github.com/vulhub/vulhub",
             "github.com/nomi-sec/poc-in-github", "github.com/trickest/cve")
DUP_HOSTS = {"exploit-db.com", "rapid7.com", "metasploit.com", "vulhub.org"}

def is_known_source(key):
    if not key:
        return False
    host = key.split("/", 1)[0]
    return host in DUP_HOSTS or any(key == repo or key.startswith(repo + "/") for repo in DUP_REPOS)

msf_ref_pairs = set()
for row in msf_compatible_rows.itertuples(index=False):
    for link in URL_PATTERN.findall(str(row.msf_references).replace("\\/", "/")):
        normalized = normalize_url(link)
        if normalized:
            msf_ref_pairs.add((row.cve_id, normalized))

df_poc["known_source_duplicate"] = df_poc["normalized_url"].map(is_known_source)
df_poc["msf_reference_duplicate"] = [(cve, key) in msf_ref_pairs for cve, key in zip(df_poc["cve_id"], df_poc["normalized_url"])]
df_poc["eligible_reference"] = (df_poc["explicit_poc_hint"] & ~df_poc["known_source_duplicate"] & ~df_poc["msf_reference_duplicate"])
# Un URL può provenire da più indici: si conserva la provenienza ma si conta una sola volta.
poc_eligible = df_poc.loc[df_poc["eligible_reference"]].copy()
poc_ids = set(poc_eligible["cve_id"])
df_poc.to_csv(PROCESSED_DIR / "poc_index_references.csv", index=False)

sources = {"vulhub": df_vulhub, "metasploit": df_msf_exploits, "exploitdb": df_edb,
           "github": df_github, "falco": df_falco}
EVIDENCE_FLAGS = ["has_vulhub", "msf_compatible", "edb_compatible", "has_poc_reference", "falco_syscall"]
poc_counts = poc_eligible.groupby("cve_id")["normalized_url"].nunique()
source_counts = {name: source.groupby("cve_id").size() for name, source in sources.items()}

def enrich(frame):
    frame = frame.copy()
    for name, counts in source_counts.items():
        frame[f"{name}_count"] = frame["cve_id"].map(counts).fillna(0).astype(int)
        frame[f"has_{name}"] = frame[f"{name}_count"].gt(0)
    for name, ids in {"msf_linux": msf_linux_ids, "edb_linux": edb_linux_ids,
                      "msf_compatible": msf_compatible_ids, "edb_compatible": edb_compatible_ids,
                      "falco_syscall": falco_syscall_ids, "has_poc_reference": poc_ids,
                      "edb_verified": edb_verified_ids, "edb_verification_known": edb_known_ids}.items():
        frame[name] = frame["cve_id"].isin(ids)
    frame["msf_best_rank"] = frame["cve_id"].map(msf_ranks)
    frame["msf_rank_known"] = frame["msf_best_rank"].notna()
    frame["poc_reference_count"] = frame["cve_id"].map(poc_counts).fillna(0).astype(int)
    frame["context_source_count"] = frame[EVIDENCE_FLAGS].sum(axis=1)
    return frame

df_enriched = enrich(df_enriched)
df_kernel_enriched = enrich(df_kernel)

evidence_tables = [
    ("vulhub_paths", df_vulhub, "vulhub_path"),
    ("msf_modules", df_msf_exploits, "msf_module"),
    ("msf_compatible_modules", msf_compatible_rows, "msf_module"),
    ("edb_ids", df_edb, "edb_id"),
    ("edb_compatible_ids", edb_compatible_rows, "edb_id"),
    ("falco_rules", df_falco, "falco_rule"),
    ("github_reference_types", df_github, "github_type"),
    ("poc_urls", poc_eligible, "poc_url"),
    ("poc_indexes", poc_eligible, "poc_index"),
]
for col, source, field in evidence_tables:
    values = source.groupby("cve_id")[field].agg(lambda s: json.dumps(sorted(set(s.dropna().astype(str))), ensure_ascii=False))
    for frame in (df_enriched, df_kernel_enriched):
        frame[col] = frame["cve_id"].map(values).fillna("[]")

# Tracciabilità dei motivi di inclusione/esclusione per ogni record NVD.
audit = df_all[["cve_id", "published", "status", "platform_nvd", "attack_vector", "kernel"]].copy()
audit["decision"] = "outside_period"
audit.loc[in_period, "decision"] = "rejected"
audit.loc[audit["cve_id"].isin(df_valid["cve_id"]), "decision"] = "attack_vector_excluded"
for reason, frame in [("kernel_separate", df_kernel), ("platform_nonlinux", df_nonlinux),
                      ("platform_conflict", df_conflicts), ("platform_unknown", df_pending),
                      ("candidate_nvd", df_initial), ("candidate_external_provisional", df_recovered)]:
    audit.loc[audit["cve_id"].isin(frame["cve_id"]), "decision"] = reason
audit["falco_syscall"] = audit["cve_id"].isin(falco_syscall_ids)
audit["vulhub_match"] = audit["cve_id"].isin(vulhub_ids)
audit["external_linux"] = audit["cve_id"].isin(msf_linux_ids | edb_linux_ids)
audit.to_csv(PROCESSED_DIR / "selection_audit.csv", index=False)
print("Casi di controllo (assenza dall'audit = ID non acquisito):")
display(audit[audit["cve_id"].isin(["CVE-2024-3094", "CVE-2020-14386"])])

for name, frame in {"candidates_enriched": df_enriched, "kernel_enriched": df_kernel_enriched,
                    "platform_pending": df_pending, "platform_conflicts": df_conflicts}.items():
    frame.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)
print(f"Candidate NVD: {len(df_initial)} | Recuperi provvisori: {len(df_recovered)}")
print(f"Conflitti: {len(df_conflicts)} | Piattaforma incerta: {len(df_pending)}")
print(f"Candidate arricchite: {len(df_enriched)} | Kernel separato: {len(df_kernel_enriched)}")
print("Copertura indicatori premiabili:")
print(df_enriched[EVIDENCE_FLAGS].sum())


## 11. Ranking documentale e soglia
Tre categorie: ambiente (30), exploit/PoC (massimo 50) e detection (20). Nella categoria exploit si usa la fonte prioritaria: Metasploit con rank interpolato 30–40; in sua assenza Exploit-DB verificato 30 / non verificato 25; in assenza di entrambi, riferimento PoC 20. Le fonti secondarie contribuiscono una volta con un bonus +5 ciascuna. Rank MSF ignoto = minimo; stato EDB ignoto = nessun premio di verifica. Il punteggio è arrotondato a due decimali prima di ordinare e selezionare. I valori sono configurabili e ispirati alla letteratura multi-fonte, non validati dai paper. Il bonus riconosce la potenziale complementarità, non l'indipendenza provata. Niente punti per tag NVD, GitHub generico o parole chiave. Il pool contiene tutte le candidate con score >= MIN_SCORE, senza limite di dimensione. Spareggio: AV Local e CVE ID.


In [ ]:
def score_candidates(frame):
    result = frame.copy()
    msf = result["msf_compatible"]
    edb = result["edb_compatible"]
    poc = result["has_poc_reference"]
    rank = result["msf_best_rank"].fillna(0)
    result["score_environment"] = result["has_vulhub"].astype(int) * WEIGHTS["vulhub"]
    result["score_msf"] = (WEIGHTS["msf_min"] + (WEIGHTS["msf_max"] - WEIGHTS["msf_min"]) * rank / 600).where(msf, 0)
    edb_primary = result["edb_verified"].map({True: WEIGHTS["edb_verified"], False: WEIGHTS["edb_unverified"]})
    result["score_edb"] = edb_primary.where(~msf, WEIGHTS["edb_bonus"]).where(edb, 0)
    result["score_poc"] = ((msf | edb).map({True: WEIGHTS["poc_bonus"], False: WEIGHTS["poc_primary"]})).where(poc, 0)
    result["score_detection"] = result["falco_syscall"].astype(int) * WEIGHTS["falco"]
    result["score_exploit"] = result[["score_msf", "score_edb", "score_poc"]].sum(axis=1)
    result["score"] = result[["score_environment", "score_exploit", "score_detection"]].sum(axis=1).round(2)
    for col in ["score_msf", "score_edb", "score_poc", "score_exploit", "score_environment", "score_detection"]:
        result[col] = result[col].round(2)
    assert result["score_exploit"].between(0, MAX_EXPLOIT + 0.01).all()
    assert result["score"].between(0, 100).all()
    return result

df_ranked = score_candidates(df_enriched)
df_ranked["local_priority"] = df_ranked["attack_vector"].eq("LOCAL").astype(int)
df_ranked = df_ranked.sort_values(["score", "local_priority", "cve_id"], ascending=[False, False, True]).reset_index(drop=True)
df_ranked["ranking"] = df_ranked.index + 1
df_pool = df_ranked[df_ranked["score"].ge(MIN_SCORE)].copy()
assert df_ranked["cve_id"].is_unique and not df_ranked["kernel"].any()
assert df_pool["score"].ge(MIN_SCORE).all()
assert len(df_pool) == int(df_ranked["score"].ge(MIN_SCORE).sum())
assert df_ranked["score"].is_monotonic_decreasing
for filename, frame in [("ranked_cves.csv", df_ranked), ("initial_pool.csv", df_pool)]:
    frame.to_csv(OUTPUT_DIR / filename, index=False)
view_cols = ["ranking", "cve_id", "attack_vector", "platform_basis", "has_vulhub", "msf_compatible", "msf_best_rank", "edb_compatible", "edb_verified", "has_poc_reference", "falco_syscall", "score"]
print(f"CVE ordinate: {len(df_ranked)} | Pool con score >= {MIN_SCORE}: {len(df_pool)}")
print(f"Massimo teorico: {MAX_SCORE}; osservato: {df_ranked['score'].max()}")
display(df_pool[view_cols])


## 12. Risultati ed esportazione
Un unico pool per soglia, un ranking locale e un manifest di esecuzione. Confronto tra soglie, coverage delle fonti, provenienza Linux e AV. Gli eventuali file storici dei vecchi pool non sono cancellati, ma non vengono più generati. Nessuna metrica di performance di Falco.


In [ ]:
print("Distribuzione AV candidate:")
print(df_ranked["attack_vector"].value_counts())
print("\nDistribuzione AV pool:")
print(df_pool["attack_vector"].value_counts())
print("\nProvenienza piattaforma nel pool:")
print(df_pool["platform_basis"].value_counts())
coverage = pd.DataFrame({"candidate": df_ranked[EVIDENCE_FLAGS].sum(), "pool": df_pool[EVIDENCE_FLAGS].sum()})
display(coverage)
print("\nDistribuzione punteggi:")
print(df_ranked["score"].value_counts().sort_index())
threshold_rows = []
for threshold in sorted(set([0, 20, 30, 40, 50, 60, 70, 80, 90, 100, MIN_SCORE])):
    selected = df_ranked[df_ranked["score"].ge(threshold)]
    threshold_rows.append({"threshold": threshold, "total": len(selected),
                           "network": int(selected["attack_vector"].eq("NETWORK").sum()),
                           "local": int(selected["attack_vector"].eq("LOCAL").sum())})
df_thresholds = pd.DataFrame(threshold_rows)
display(df_thresholds)

def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

input_files = list(NVD_FILES) + [vulhub_file, msf_file, edb_file] + [RAW_DIR / name for name in FALCO_FILES.values()] + poc_input_files
manifest = {
    "method_version": "4.0", "code_revision": "4.1-refactor", "run_utc": datetime.now(timezone.utc).isoformat(),
    "score_max_theoretical": MAX_SCORE, "min_score": MIN_SCORE,
    "ranking_tiebreak": ["local_priority desc", "cve_id asc"],
    "literature": ["10.1145/3475960.3475985", "10.1016/j.jss.2023.111679", "10.1093/cybsec/tyaa015", "https://www.usenix.org/conference/usenixsecurity18/presentation/mu", "arXiv:2509.01835v2"],
    "start_date": START_DATE, "end_date": END_DATE, "attack_vectors": ATTACK_VECTORS,
    "kernel_separate": True, "weights": WEIGHTS, "poc_indexes": poc_index_status,
    "counts": {"nvd_unique": len(df_all), "valid_in_period": len(df_valid), "linux_nvd_before_av": len(df_linux_pre_av),
               "kernel": len(df_kernel), "av_non_kernel": len(df_av), "nvd_linux": len(df_initial),
               "recovered": len(df_recovered), "pending": len(df_pending), "conflicts": len(df_conflicts),
               "enriched": len(df_enriched), "positive_score": int(df_ranked["score"].gt(0).sum()), "pool": len(df_pool),
               "pool_network": int(df_pool["attack_vector"].eq("NETWORK").sum()), "pool_local": int(df_pool["attack_vector"].eq("LOCAL").sum()),
               "pool_external_provisional": int(df_pool["platform_basis"].eq("external_provisional").sum())},
    "coverage": {key: {k: int(v) for k,v in values.items()} for key, values in coverage.to_dict().items()},
    "thresholds": threshold_rows,
    "inputs": [{"file": p.relative_to(BASE_DIR).as_posix(), "bytes": p.stat().st_size, "sha256": file_hash(p)} for p in input_files],
    "limitations": ["NVD feed annuali per anno ID", "CPE e kernel euristici; recupero provvisorio", "Compatibilità documentale, non testata", "PoC indicizzati con indizio esplicito: falsi positivi e falsi negativi possibili", "Bonus non dimostra indipendenza", "Pesi e soglia empirici", "Nessun log o test Falco; regole pubbliche non equivalgono a rilevabilità", "Possibile sovrarappresentazione delle CVE applicative ben documentate"]
}
(OUTPUT_DIR / "run_summary.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nOutput V4 disponibili in: {OUTPUT_DIR}")
